In [1]:
%load_ext autoreload
%autoreload 2

import matplotlib.pyplot as plt
import polars as pl

from ethos.constants import PROJECT_ROOT
from ethos.constants import SpecialToken as ST
from ethos.inference.constants import Reason, Task
from ethos.metrics import compute_and_print_metrics, preprocess_inference_results

results_dir = PROJECT_ROOT / "results"

## ICU Admission results

In [ ]:
def process_icu_admission_results(input_dir, **kwargs) -> pl.DataFrame:
    return preprocess_inference_results(
        input_dir,
        actual_expr=pl.col("actual").is_in([ST.ICU_ADMISSION]),
        expected_expr=pl.col("expected").is_in([ST.ICU_ADMISSION]),
        filter_ambiguous=pl.col("stop_reason") == Reason.GOT_TOKEN,
        **kwargs,
    )

In [2]:
# DEBUGGING ONLY
import sys
from pathlib import Path

project_root = Path.cwd()
while not (project_root / 'pyproject.toml').exists() and project_root != project_root.parent:
    project_root = project_root.parent

sys.path.insert(0, str(project_root))
from src.ethos.datasets.mimic_icu import ICUAdmissionDataset
path = "../../../../mnt/data_share/project_henri/ethos-ares/mimic-tokenized/test"
dataset = ICUAdmissionDataset(path)
sum_adm = sum([True if d[1]['expected'] == 'ICU_ADMISSION' else False for d in dataset])
sum_adm#/len(dataset)

64588 64588


9956

In [8]:
import json
true_ids = [d[1]['hadm_id'] for d in dataset if d[1]['expected'] == 'ICU_ADMISSION']
with open("true_ids.json", "w") as f:
    json.dump(true_ids, f)


In [11]:
for i in range(len(dataset)): 
    if dataset[i][1]['hadm_id'] == 29880150.0:
        print(i, dataset[i][1])
        break

515 {'expected': 'HOSPITAL_DISCHARGE', 'true_token_dist': 43, 'true_token_time': 20182000000, 'icu_stay_id': None, 'patient_id': 10260936, 'hadm_id': 29880150, 'prediction_time': 5387335418000000, 'data_idx': 441837}


In [6]:
import pandas as pd
df_token = pd.read_parquet(f"{path}/13_inject_time_intervals/0.parquet")

In [7]:
df_21472555 = df_token[df_token['hadm_id'].isin([23747233,25248373])]
df_21472555[df_21472555.code.str.contains('HOSPITAL') | df_21472555.code.str.contains('ICU_') | df_21472555.code.str.contains('DEATH')]

,subject_id,time,code,numeric_value,text_value,hadm_id,edstay_id,icustay_id,insurance,language,marital_status,race,drg_severity,drg_mortality,emar_id,emar_seq,priority
11572741,16091960,2140-09-02 20:35:00,HOSPITAL_ADMISSION,NaN,OBSERVATION_ADMIT,25248373.0,NaN,NaN,Other,?,SINGLE,ASIAN,NaN,NaN,NaN,NaN,NaN
11573390,16091960,2140-09-05 08:27:00,HOSPITAL_ADMISSION,NaN,EW_EMER,23747233.0,NaN,NaN,Other,?,SINGLE,ASIAN,NaN,NaN,NaN,NaN,NaN
11573458,16091960,2140-09-05 11:52:26,HOSPITAL_DISCHARGE,NaN,HOME,25248373.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
11573673,16091960,2140-09-05 20:56:14,ICU_ADMISSION,NaN,ICU_TYPE//Medical/Surgical Intensive Care Unit...,23747233.0,NaN,35652570.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
11573674,16091960,2140-09-05 20:56:14,ICU_TYPE//MEDICAL/SURGICAL_INTENSIVE_CARE_UNIT...,NaN,ICU_TYPE//Medical/Surgical Intensive Care Unit...,23747233.0,NaN,35652570.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
11573933,16091960,2140-09-06 15:47:37,ICU_DISCHARGE,NaN,ICU_TYPE//Medical/Surgical Intensive Care Unit...,23747233.0,NaN,35652570.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
11574739,16091960,2140-09-10 18:30:59,HOSPITAL_DISCHARGE,NaN,HOME HEALTH CARE,23747233.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [16]:
df2 = df_token[df_token['hadm_id'] == 21430309.0]
df2[df2.code.str.contains('DRG')]
# df_token.iloc[11573392+66]

,subject_id,time,code,numeric_value,text_value,hadm_id,edstay_id,icustay_id,insurance,language,marital_status,race,drg_severity,drg_mortality,emar_id,emar_seq,priority
5755608,13100208,2157-05-20 13:24:00,DRG//65,NaN,NaN,21430309.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
icu_admission_results = []
for input_dir in (results_dir / Task.ICU_ADMISSION).iterdir():
    df = process_icu_admission_results(input_dir)
    rep_num = df["counts"].mean()
    res = compute_and_print_metrics(
        df["expected"],
        df["actual"],
        f"ICU Admission (rep_num={rep_num:.2f})\n{input_dir.name}",
    )
    icu_admission_results.append(
        {
            "name": input_dir.name,
            "auc": res["fitted_auc"],
            "rep_num": rep_num,
        }
    )
    plt.show()
icu_admission_results = pl.DataFrame(icu_admission_results).sort("auc", descending=True)

In [ ]:
icu_admission_results